# Chapter 13: GraphQL and Building APIs

**Level 3: Integration & Engineering** · ⏱ about 3 hours · Dataset: *EduGraph*

### What you will learn

1. Three ways to put a knowledge graph behind an **API**, and when to use each
2. **Stardog GraphQL**: queries generated from your data, nesting, arguments, variables, and **scoping to named graphs** (with one trap)
3. **Stored queries**: SPARQL saved on the server under a name, shared by every client
4. **Hands-on:** a **FastAPI** REST service over stored queries, with `/students/{id}/gaps`, input validation and proper error codes, run in-process and as a real HTTP server
5. Design rules for graph-backed APIs: parameters, connections, configuration, paging and security

Chapter 14 adds tests, users and deployment, which completes **Mini-project 3: the EduGraph API**.

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
TUTORIAL = Path('..').resolve()
for p in (SRC, str(TUTORIAL)):
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import requests
import stardog
from rdflib import URIRef

from kg import client, imports, sparql, stored_queries

pd.set_option('display.max_colwidth', 90)
conn = client.connect()
PREFIX = 'urn:tutorial:ch13'
EDU = TUTORIAL / 'data' / 'edugraph'
CURRICULUM = f'{PREFIX}:curriculum'
SIS = {n: f'{PREFIX}:sis:{n}' for n in ('students', 'enrollments', 'assessments')}
GRAPHS = [CURRICULUM, *SIS.values()]

with client.transaction(conn):
    conn.clear(graph_uri=CURRICULUM)
    conn.add(stardog.content.File(str(EDU / 'curriculum.ttl')), graph_uri=CURRICULUM)
for name, graph in SIS.items():
    imports.import_file(EDU / 'sis' / f'{name}.csv', TUTORIAL / 'ontology' / 'mappings' / f'{name}.sms', graph)
print('EduGraph loaded into', len(GRAPHS), 'graphs')

EduGraph loaded into 4 graphs


## 13.1 Three ways to serve a graph

| | **SPARQL endpoint** | **GraphQL** | **Custom REST API** |
|---|---|---|---|
| What clients send | Any SPARQL query | A GraphQL query (a nested field list) | `GET /students/S023/gaps` |
| Built by | Stardog, out of the box (`/{db}/query`) | Stardog, out of the box (`/{db}/graphql`) | You (FastAPI here) |
| Clients need to know | SPARQL and the model | GraphQL and the field names | Nothing about graphs |
| Flexibility | Anything | Tree-shaped reads | Only the endpoints you design |
| Control, safety, caching | Low: any query can run | Medium | High: fixed, reviewed queries |
| Good for | Analysts, data scientists, other graph tools | Front-end developers | Apps, partners, public APIs |

Most real systems use more than one: analysts get SPARQL, the web app gets GraphQL or REST.

## 13.2 Stardog GraphQL

Stardog answers **GraphQL** queries directly from the data. There's no schema to write first: type and field names come from your IRIs.

- `edu:Topic` becomes the type `edu_Topic`, and `edu:name` becomes the field `edu_name` (prefix, underscore, local name).
- The `@prefix(edu: "…")` directive declares the prefix inside the query.
- Arguments **filter**: `edu_Topic(edu_level: 1)` means topics whose level is 1.
- Nested fields **follow links**: `edu_requires { edu_name }`.
- Aliases (`name: edu_name`) rename fields in the result.

pystardog sends GraphQL with `conn.graphql().query(text, variables)`:

In [2]:
gql = conn.graphql()

def graphql(query, variables=None):
    return gql.query(query, variables)

beginner = graphql(f'''
query @prefix(edu: "http://example.org/edu#") @config(graph: {json.dumps(GRAPHS)}) {{
  edu_Topic(edu_level: 1) {{ name: edu_name  subject: edu_subject }}
}}''')
pd.DataFrame(beginner)

,name,subject
0,Python basics,Programming
1,Control flow,Programming
2,Functions,Programming
3,Algebra refresher,Mathematics
4,SQL basics,Data


### Nesting follows the graph

GraphQL's strength is **tree-shaped** results in one request: a topic, its prerequisites, and *their* prerequisites:

In [3]:
tree = graphql(f'''
query @prefix(edu: "http://example.org/edu#") @config(graph: {json.dumps(GRAPHS)}) {{
  edu_Topic(edu_name: "Linear regression") {{
    name: edu_name
    requires: edu_requires {{
      name: edu_name
      requires: edu_requires {{ name: edu_name }}
    }}
  }}
}}''')
print(json.dumps(tree, indent=2))

[
  {
    "name": "Linear regression",
    "requires": [
      {
        "name": "Linear algebra",
        "requires": {
          "name": "Algebra refresher"
        }
      },
      {
        "name": "Statistics",
        "requires": {
          "name": "Probability"
        }
      },
      {
        "name": "pandas",
        "requires": {
          "name": "NumPy"
        }
      }
    ]
  }
]


### Variables

Like SPARQL bindings, GraphQL **variables** keep values out of the query text:

In [4]:
BY_LEVEL = f'''
query topicsByLevel($level: Int) @prefix(edu: "http://example.org/edu#") @config(graph: {json.dumps(GRAPHS)}) {{
  edu_Topic(edu_level: $level) {{ name: edu_name }}
}}'''
for level in (1, 3):
    print(level, [t['name'] for t in graphql(BY_LEVEL, {'level': level})])

1 ['Python basics', 'Control flow', 'Functions', 'Algebra refresher', 'SQL basics']


3 ['Knowledge graphs', 'Linear regression', 'Model evaluation', 'Clustering', 'Neural networks', 'Classification']


### Scope GraphQL to your graphs, and a trap

Unlike SPARQL (which reads only the default graph unless told otherwise), Stardog GraphQL reads **all graphs** of the database when no graph is configured. A quick test with two small graphs:

In [5]:
A, B = f'{PREFIX}:scope-test:a', f'{PREFIX}:scope-test:b'
with client.transaction(conn):
    for g, label in ((A, 'in graph A'), (B, 'in graph B')):
        conn.clear(graph_uri=g)
        conn.add(stardog.content.Raw(f'<urn:{label[-1]}> a <http://example.org/edu#ScopeTest> ; '
                                     f'<http://example.org/edu#name> "{label}" .'.encode(), 'text/turtle'), graph_uri=g)

for label, config in [('no @config', ''), ('@config(graph: A)', f'@config(graph: "{A}")'),
                      ('@config(default_graph: A)', f'@config(default_graph: "{A}")')]:
    rows = graphql(f'query @prefix(edu: "http://example.org/edu#") {config} {{ edu_ScopeTest {{ edu_name }} }}')
    print(f'{label:28} -> {[r["edu_name"] for r in rows]}')

no @config                   -> ['in graph A', 'in graph B']


@config(graph: A)            -> ['in graph A']


@config(default_graph: A)    -> ['in graph A', 'in graph B']


- **No config** → both graphs, i.e. everything in the database.
- **`@config(graph: …)`** → only the listed graph(s). ✅
- **`@config(default_graph: …)`** → **silently ignored**: both graphs again. It looks right and does nothing, so test your scoping.

In an application, always set `graph`, or better, enforce access on the server with named-graph security (Chapter 14).

> 🔁 **Neo4j equivalent:** the Neo4j GraphQL Library generates a GraphQL API from type definitions you write (`type Topic { name: String, requires: [Topic!]! @relationship(…) }`). Stardog derives it from the data, and you can still upload a schema to control it.

## 13.3 Stored queries

A **stored query** is SPARQL saved **on the server** under a name. Clients run it by name, with bindings, instead of sending the text. Benefits:

- **One source of truth**: the web app, Studio users, reports and other services run the same reviewed query.
- **Change without redeploying clients**: fix the query on the server, and every caller gets the fix.
- **Visible to admins** (Studio lists them) and subject to **permissions** (Chapter 14).

`kg.stored_queries.publish()` stores every `.rq` file in a folder, prefixed so tutorial queries are easy to spot and remove:

In [6]:
print(Path(stored_queries.__file__).read_text())

"""Publish .rq files as Stardog stored queries, and remove them again (Chapter 13).

    names = stored_queries.publish('edugraph', prefix='tutorial_edugraph_')
    sparql.run_stored(conn, 'tutorial_edugraph_prerequisite_gaps', graphs=..., threshold=50)
"""
from kg import client
from kg.sparql import QUERY_DIR


def list_names(prefix: str = '') -> list[str]:
  with client.admin() as admin:
    return sorted(q.name for q in admin.stored_queries() if q.name.startswith(prefix))


def publish(folder: str, prefix: str) -> list[str]:
  """Store every src/kg/queries/<folder>/*.rq on the server as <prefix><file name>. Replaces existing ones."""
  settings = client.load_settings()
  names = []
  with client.admin(settings) as admin:
    existing = {q.name for q in admin.stored_queries()}
    for path in sorted((QUERY_DIR / folder).glob('*.rq')):
      name = prefix + path.stem
      if name in existing:
        admin.stored_query(name).delete()          # pystardog has no update: delete and re-

In [7]:
QUERY_PREFIX = 'tutorial_edugraph_'
published = stored_queries.publish('edugraph', QUERY_PREFIX)
print('published:', *published, sep='\n  ')
print('\nall stored queries on the server:', stored_queries.list_names())

published:
  tutorial_edugraph_course_performance
  tutorial_edugraph_prerequisite_gaps
  tutorial_edugraph_student_profile
  tutorial_edugraph_topic_difficulty
  tutorial_edugraph_topic_prerequisites
  tutorial_edugraph_topics



all stored queries on the server: ['start', 'tutorial_edugraph_course_performance', 'tutorial_edugraph_prerequisite_gaps', 'tutorial_edugraph_student_profile', 'tutorial_edugraph_topic_difficulty', 'tutorial_edugraph_topic_prerequisites', 'tutorial_edugraph_topics']


Running a stored query: the **name** replaces the query text; bindings and graphs work as before. `sparql.run_stored()` is the stored-query twin of `run_query()`:

In [8]:
sparql.run_stored(conn, QUERY_PREFIX + 'topic_prerequisites', graphs=GRAPHS,
                  topic=URIRef('http://example.org/edu/topic/regression'))

,id,name,level,direct
0,linear-algebra,Linear algebra,2,True
1,statistics,Statistics,2,True
2,pandas,pandas,2,True
3,data-structures,Data structures,2,False
4,numpy,NumPy,2,False
5,probability,Probability,2,False
6,arithmetic,Algebra refresher,1,False
7,control-flow,Control flow,1,False
8,functions,Functions,1,False
9,python-basics,Python basics,1,False


In Studio, stored queries are listed with the database, where analysts can open and run them.

## 13.4 Hands-on: the EduGraph REST API

`api/main.py` is a small FastAPI app. Every endpoint runs a **stored query** with **bindings**:

In [9]:
print((TUTORIAL / 'api' / 'main.py').read_text())

r"""EduGraph REST API (Chapter 13): a thin FastAPI layer over SPARQL queries stored in Stardog.

Run it from the stardog_tutorial folder:
    ..\.venv\Scripts\uvicorn api.main:app --reload
then open http://127.0.0.1:8000/docs

Settings (environment variables, all optional):
    EDUGRAPH_GRAPHS         comma-separated named graphs to query (default: Chapter 13's graphs)
    EDUGRAPH_QUERY_PREFIX   prefix of the stored queries (default: tutorial_edugraph_)
"""
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

import pandas as pd                                                     # noqa: E402
from fastapi import Depends, FastAPI, HTTPException, Path as PathParam, Query   # noqa: E402
from rdflib import URIRef                                               # noqa: E402

from kg import client, sparql                                           # noqa: E402

DEFAULT_GRAPHS = ['urn:tutorial:ch13:curriculum', 'urn:tutorial:ch13:s

### What the code does, and why

| Concern | How |
|---|---|
| Queries | Stored on the server, called by name (`run_stored`); no SPARQL text in the API |
| User input | Path and query parameters are **validated** by FastAPI (`pattern=r'^S[0-9]{3}$'`, `ge=0, le=100`) and passed as **bindings**, so no injection is possible |
| Errors | Unknown student or topic → **404**; invalid input → **422** (automatic) |
| Connections | One per request via a FastAPI dependency (`Depends(connection)`), closed afterwards, because pystardog connections aren't thread-safe |
| Configuration | Graphs and query prefix from environment variables, so the same code serves test and production graphs |
| JSON | Typed values from `run_stored` (ints, dates), with missing values as `null` |

### Try it in-process

FastAPI's `TestClient` calls the app without starting a server, which is perfect for notebooks and tests:

In [10]:
from fastapi.testclient import TestClient
from api.main import app

api = TestClient(app)

def call(url):
    r = api.get(url)
    print(f'GET {url}  ->  {r.status_code}')
    return r.json()

call('/health')

GET /health  ->  200


{'status': 'ok', 'database': 'mysampledb', 'triples': 4479}

In [11]:
gaps = call('/students/S023/gaps?threshold=50')
print(json.dumps(gaps, indent=2))

GET /students/S023/gaps?threshold=50  ->  200
{
  "id": "S023",
  "threshold": 50,
  "gaps": [
    {
      "student_name": "Aarav Sharma",
      "weak_topic": "Statistics",
      "score": 33,
      "course": "ML201",
      "blocks": "Classification, Linear regression, Neural networks, Model evaluation"
    }
  ]
}


In [12]:
pd.DataFrame(call('/topics/regression/prerequisites')['prerequisites'])

GET /topics/regression/prerequisites  ->  200


,id,name,level,direct
0,linear-algebra,Linear algebra,2,True
1,statistics,Statistics,2,True
2,pandas,pandas,2,True
3,data-structures,Data structures,2,False
4,numpy,NumPy,2,False
5,probability,Probability,2,False
6,arithmetic,Algebra refresher,1,False
7,control-flow,Control flow,1,False
8,functions,Functions,1,False
9,python-basics,Python basics,1,False


And the error cases:

In [13]:
for url in ['/students/S999', '/students/robert', '/students/S023/gaps?threshold=500', '/topics/astrology/prerequisites']:
    print(json.dumps(call(url))[:180], '\n')

GET /students/S999  ->  404
{"detail": "No student 'S999'"} 

GET /students/robert  ->  422
{"detail": [{"type": "string_pattern_mismatch", "loc": ["path", "student_id"], "msg": "String should match pattern '^S[0-9]{3}$'", "input": "robert", "ctx": {"pattern": "^S[0-9]{3} 

GET /students/S023/gaps?threshold=500  ->  422
{"detail": [{"type": "less_than_equal", "loc": ["query", "threshold"], "msg": "Input should be less than or equal to 100", "input": "500", "ctx": {"le": 100}}]} 



GET /topics/astrology/prerequisites  ->  404
{"detail": "No topic 'astrology'"} 



### Run it as a real server

The same app, served by **uvicorn** over HTTP. From a terminal in `stardog_tutorial/`:

```
..\.venv\Scripts\uvicorn api.main:app --reload
```

and then open **http://127.0.0.1:8000/docs** for interactive documentation. Here we start it in the background on port 8765, call it with `requests` like any client would, and stop it:

In [14]:
PORT = 8765
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api.main:app', '--port', str(PORT)],
                          cwd=TUTORIAL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
BASE = f'http://127.0.0.1:{PORT}'
try:
    for _ in range(60):                                   # wait until it answers
        try:
            requests.get(f'{BASE}/health', timeout=2)
            break
        except requests.ConnectionError:
            time.sleep(0.5)
    print(requests.get(f'{BASE}/students/S023', timeout=30).json())
    print(requests.get(f'{BASE}/courses', timeout=30).json()[0])
    spec = requests.get(f'{BASE}/openapi.json', timeout=30).json()
    print('\nOpenAPI paths:', list(spec['paths']))
finally:
    server.terminate()
    server.wait(timeout=10)
    print('server stopped')

{'id': 'S023', 'name': 'Aarav Sharma', 'city': 'Sao Paulo', 'year': 2, 'courses': 'MA120, ML201', 'assessments': 9, 'avg_score': 60.0}


{'course': 'DB150', 'title': 'Databases and Knowledge Graphs', 'students': 24, 'assessments': 87, 'avg_score': 66.0}

OpenAPI paths: ['/health', '/topics', '/topics/{topic_id}/prerequisites', '/students/{student_id}', '/students/{student_id}/gaps', '/courses']
server stopped


FastAPI generates the **OpenAPI** description (`/openapi.json`) from the code: parameter types, patterns and limits included. That's what `/docs` renders, and what client generators consume.

## 13.5 Design rules for graph-backed APIs

1. **Design endpoints around questions**, not around the graph. `/students/{id}/gaps` beats exposing triples.
2. **Keep SPARQL out of the API code**: in `.rq` files, published as **stored queries**, reviewed like code.
3. **Validate every input, then bind it.** Never build query text from request data (Chapter 6).
4. **Configure the graphs** per environment (test, staging, production), and never let clients choose them freely.
5. **Page** large results (`limit` / `offset` → `run_stored(..., limit=…)`) and **cache** stable answers (topics change rarely).
6. **Authenticate** the API's users, and give the API itself a Stardog user with **only the permissions it needs**: read on the graphs it serves, execute on its stored queries. Chapter 14 builds that.

## 13.6 Exercises

**Exercise 1.** Write a GraphQL query for each **course** with its code, title and the **names** of the topics it covers.

<details><summary>Solution</summary>

```python
graphql(f'''query @prefix(edu: "http://example.org/edu#") @config(graph: {json.dumps(GRAPHS)}) {{
  edu_Course {{ code: edu_code  title: edu_title  covers: edu_covers {{ name: edu_name }} }} }}''')
```
</details>

**Exercise 2.** Add an endpoint `GET /topics/{topic_id}/dependents` that lists the topics that (directly or indirectly) **depend on** a topic. Write the `.rq` file (hint: `?dependent edu:requires+ ?topic`), publish it, add the route, and test it with `TestClient`.

<details><summary>Solution</summary>

`src/kg/queries/edugraph/topic_dependents.rq`:
```sparql
# Topics that depend on ?topic, directly or indirectly.
# Parameters: ?topic (IRI)
PREFIX edu: <http://example.org/edu#>
SELECT DISTINCT ?id ?name WHERE {
  ?d edu:requires+ ?topic ; edu:name ?name  BIND(STRAFTER(STR(?d), "/topic/") AS ?id) } ORDER BY ?name
```
In `api/main.py`:
```python
@app.get('/topics/{topic_id}/dependents')
def dependents(topic_id: str = PathParam(pattern=r'^[a-z0-9-]+$'), conn=Depends(connection)):
  return run(conn, 'topic_dependents', topic=URIRef(TOPIC + topic_id))
```
Then `stored_queries.publish('edugraph', QUERY_PREFIX)`, restart the kernel (so `api.main` is re-imported), and `call('/topics/statistics/dependents')`.
</details>

**Exercise 3.** Add paging to `/topics`: `?limit=5&offset=0`. What must the query have for paging to be stable?

<details><summary>Solution</summary>

Add `limit: int = Query(20, ge=1, le=100), offset: int = Query(0, ge=0)` and pass them on: `run_stored(..., limit=limit)` plus an `offset` (`conn.select(..., offset=offset)`). The query needs a deterministic `ORDER BY` (`topics.rq` has `ORDER BY ?level ?name`), otherwise pages can overlap or skip rows.
</details>

**Exercise 4.** A partner asks for "just give us the SPARQL endpoint". What are the risks compared with the REST API?

<details><summary>Solution</summary>

They could run **any** query: expensive ones (performance risk for everyone), ones that read graphs they shouldn't see (unless permissions are strict), and their code breaks whenever the model changes. The REST API exposes a small, reviewed, validated set of questions, and the model can change behind it.
</details>

In [15]:
# your answers here

## 13.7 Clean up

Removes the chapter's graphs **and** the stored queries it published.

In [16]:
CLEAN_UP = True

if CLEAN_UP:
    print('stored queries removed:', stored_queries.remove(QUERY_PREFIX))
    with client.transaction(conn):
        for g in [*GRAPHS, A, B]:
            conn.clear(graph_uri=g)
    print('graphs removed; stored queries left on the server:', stored_queries.list_names())
conn.close()

stored queries removed: ['tutorial_edugraph_course_performance', 'tutorial_edugraph_prerequisite_gaps', 'tutorial_edugraph_student_profile', 'tutorial_edugraph_topic_difficulty', 'tutorial_edugraph_topic_prerequisites', 'tutorial_edugraph_topics']


graphs removed; stored queries left on the server: ['start']


## Summary

- Serve a graph through a **SPARQL endpoint** (analysts), **GraphQL** (front ends) or a **custom REST API** (apps and partners), often several at once.
- **Stardog GraphQL** maps IRIs to fields (`edu_name`), filters with arguments, nests along links, and supports variables. It reads **all graphs** unless you set `@config(graph: …)`; `default_graph` is silently ignored.
- **Stored queries** keep SPARQL on the server, shared, reviewable and permission-controlled, and run by name with bindings (`sparql.run_stored`).
- A **FastAPI** layer turns stored queries into clean endpoints, with validation (422), not-found handling (404), one connection per request, and configuration from the environment. `TestClient` tests it in-process; uvicorn serves it for real, with OpenAPI docs.

**Next: Chapter 14, Administration and DevOps.** Users, roles and permissions (a least-privilege user for the API), named-graph security, database options and backups, automated deployment, and pytest tests with a temporary named graph per test. It completes Mini-project 3.